In [3]:
"""
Men's NCAA Tournament - Feature Selection & Hyperparameter Optimization
Run this in Jupyter notebook for each round model.

Change ROUND_CONFIG at the top to switch between rounds.
"""

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (train_test_split, cross_val_score,
                                      StratifiedKFold, RandomizedSearchCV)
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score, accuracy_score, brier_score_loss
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)

# ============================================================================
# CONFIGURATION — change this for each round
# ============================================================================
ROUND_CONFIG = {
    'name': 'Round 1',
    'filter': ['Second Round'],          # round values to filter on
    'test_size': 0.15,                   # test split
    'val_size': 0.18,                    # validation split (from remaining)
    'model_type': 'xgboost',            # 'xgboost' or 'random_forest'
}

# Other round configs (uncomment one at a time):

#ROUND_CONFIG = {
#     'name': 'Round 2',
#     'filter': ['Second Round'],
#     'test_size': 0.20,
#     'val_size': None,                  # no separate val split
#     'model_type': 'random_forest',
#}

#ROUND_CONFIG = {
#     'name': 'Weekend 2',
#     'filter': ['Sweet 16', 'Elite Eight'],
#     'test_size': 0.25,
#     'val_size': None,
#     'model_type': 'random_forest',
# }

#ROUND_CONFIG = {
#     'name': 'Weekend 3',
#     'filter': ['Final Four', 'Championship'],
#     'test_size': 0.30,
#     'val_size': None,
#     'model_type': 'logistic_regression',
#}

# ============================================================================
# LOAD DATA
# ============================================================================
df = pd.read_csv('men_2027_training_round2.csv')

exclude = ['Unnamed: 0', 'game_id', 'year', 'region', 'round',
           'high_bracket_team', 'low_bracket_team',
           'high_bracket_seed', 'low_bracket_seed', 'seed', 'win']
all_features = [c for c in df.columns if c not in exclude]

rnd = df[df['round'].isin(ROUND_CONFIG['filter'])].copy()
X = rnd[all_features].fillna(rnd[all_features].median())
y = rnd['win']

print("=" * 80)
print(f"{ROUND_CONFIG['name']} FEATURE SELECTION & HYPERPARAMETER OPTIMIZATION")
print(f"  {len(rnd)} games, {len(all_features)} features")
print(f"  Model type: {ROUND_CONFIG['model_type']}")
print("=" * 80)

# ============================================================================
# TRAIN / VAL / TEST SPLIT
# ============================================================================
if ROUND_CONFIG['val_size']:
    X_temp, X_test, y_temp, y_test = train_test_split(
        X, y, test_size=ROUND_CONFIG['test_size'], random_state=42, stratify=y)
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp, test_size=ROUND_CONFIG['val_size'], random_state=42, stratify=y_temp)
    print(f"\n  Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")
else:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=ROUND_CONFIG['test_size'], random_state=42, stratify=y)
    X_val, y_val = None, None
    print(f"\n  Train: {len(X_train)}, Test: {len(X_test)}")

sc = StandardScaler()
X_train_s = pd.DataFrame(sc.fit_transform(X_train), columns=all_features, index=X_train.index)
X_test_s = pd.DataFrame(sc.transform(X_test), columns=all_features, index=X_test.index)
if X_val is not None:
    X_val_s = pd.DataFrame(sc.transform(X_val), columns=all_features, index=X_val.index)

# ============================================================================
# PART 1: FEATURE SELECTION (3 methods + consensus + greedy)
# ============================================================================
print("\n" + "=" * 80)
print("PART 1: FEATURE SELECTION")
print("=" * 80)

# Method 1: RF Gini Importance
print("\n--- Method 1: RF Gini Importance ---")
rf_fs = RandomForestClassifier(n_estimators=500, max_depth=8, random_state=42, n_jobs=-1)
rf_fs.fit(X_train_s, y_train)
gini_imp = pd.Series(rf_fs.feature_importances_, index=all_features).sort_values(ascending=False)
for f, v in gini_imp.head(15).items():
    print(f"  {f}: {v:.4f}")

# Method 2: Permutation Importance
print("\n--- Method 2: Permutation Importance ---")
perm = permutation_importance(rf_fs, X_test_s, y_test, n_repeats=15, random_state=42, n_jobs=-1)
perm_imp = pd.Series(perm.importances_mean, index=all_features).sort_values(ascending=False)
for f, v in perm_imp.head(15).items():
    print(f"  {f}: {v:.4f}")

# Method 3: Mutual Information
print("\n--- Method 3: Mutual Information ---")
mi = mutual_info_classif(X_train_s, y_train, random_state=42)
mi_imp = pd.Series(mi, index=all_features).sort_values(ascending=False)
for f, v in mi_imp.head(15).items():
    print(f"  {f}: {v:.4f}")

# Consensus ranking
print("\n--- Consensus Ranking (Top 30) ---")
gini_rank = gini_imp.rank(ascending=False)
perm_rank = perm_imp.rank(ascending=False)
mi_rank = mi_imp.rank(ascending=False)
avg_rank = ((gini_rank + perm_rank + mi_rank) / 3).sort_values()

for i, (f, v) in enumerate(avg_rank.head(30).items()):
    g = int(gini_rank[f]); p = int(perm_rank[f]); m = int(mi_rank[f])
    print(f"  {i+1:2d}. {f:45s} avg={v:5.1f}  (G={g:3d} P={p:3d} MI={m:3d})")

# Greedy selection (correlation threshold 0.7)
print("\n--- Greedy Selection (|r| < 0.7) ---")
selected = []
for f in avg_rank.index:
    if len(selected) >= 25:
        break
    too_corr = False
    for s in selected:
        if abs(X_train[f].corr(X_train[s])) > 0.7:
            too_corr = True
            break
    if not too_corr:
        selected.append(f)
        print(f"  {len(selected):2d}. {f}")

# Correlations among selected
print("\n--- Correlations Among Selected (|r| > 0.5) ---")
top15 = selected[:15]
corr = X_train[top15].corr()
for i in range(len(top15)):
    for j in range(i+1, len(top15)):
        r = corr.iloc[i, j]
        if abs(r) > 0.5:
            print(f"  {top15[i]:40s} vs {top15[j]:40s}: {r:.4f}")

# ============================================================================
# PART 2: EVALUATE SUBSETS
# ============================================================================
print("\n" + "=" * 80)
print("PART 2: EVALUATE FEATURE SUBSETS")
print("=" * 80)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Default RF params for evaluation
rf_eval = dict(n_estimators=300, max_depth=8, min_samples_split=5,
               min_samples_leaf=2, max_features=0.4, random_state=42, n_jobs=-1)

print(f"\n{'Feats':>6s}  {'CV ACC':>8s}  {'Test ACC':>9s}  {'Train AUC':>10s}  {'Test AUC':>9s}  {'Gap':>7s}  {'Prob Range':>15s}")
print("-" * 75)

for n in [3, 6, 9, 12, 15, 18, 21, 24, 27, 30]:
    if n > len(selected):
        break
    sub = selected[:n]
    scores = cross_val_score(RandomForestClassifier(**rf_eval), X_train_s[sub], y_train, cv=cv, scoring='accuracy')
    rf_t = RandomForestClassifier(**rf_eval)
    rf_t.fit(X_train_s[sub], y_train)
    tr_p = rf_t.predict_proba(X_train_s[sub])[:, 1]
    te_p = rf_t.predict_proba(X_test_s[sub])[:, 1]
    tr_auc = roc_auc_score(y_train, tr_p)
    te_auc = roc_auc_score(y_test, te_p)
    te_acc = accuracy_score(y_test, rf_t.predict(X_test_s[sub]))
    print(f"Top {n:2d}   {scores.mean():.4f}    {te_acc:.4f}     {tr_auc:.4f}     {te_auc:.4f}   {tr_auc-te_auc:+.4f}   [{te_p.min():.3f}, {te_p.max():.3f}]")

# ============================================================================
# PART 3: HYPERPARAMETER OPTIMIZATION
# ============================================================================
print("\n" + "=" * 80)
print("PART 3: HYPERPARAMETER OPTIMIZATION")
print("=" * 80)

# *** SET YOUR CHOSEN FEATURES HERE AFTER REVIEWING PART 2 ***
# chosen_features = selected[:7]  # example: top 7
chosen_features = selected[:12]
print(f"\nOptimizing with {len(chosen_features)} features: {chosen_features}")

if ROUND_CONFIG['model_type'] == 'xgboost':
    param_dist = {
        'max_depth': [2, 3, 4, 5, 6],
        'learning_rate': [0.01, 0.02, 0.05, 0.1, 0.15],
        'n_estimators': [100, 150, 200, 250, 300, 400],
        'min_child_weight': [1, 2, 3, 5, 7, 10],
        'subsample': [0.3, 0.4, 0.5, 0.6, 0.7, 0.8],
        'colsample_bytree': [0.3, 0.4, 0.5, 0.6, 0.7, 0.8],
        'colsample_bylevel': [0.4, 0.5, 0.6, 0.7, 0.8],
        'colsample_bynode': [0.3, 0.4, 0.5, 0.6, 0.7],
        'gamma': [0, 0.05, 0.1, 0.2, 0.5, 1.0],
        'reg_alpha': [0, 0.01, 0.1, 0.5, 1.0, 2.0],
        'reg_lambda': [1.0, 2.0, 3.0, 5.0, 7.0, 10.0],
        'max_delta_step': [0, 1, 2, 3],
        'scale_pos_weight': [0.8, 0.9, 1.0, 1.1, 1.2],
    }
    base_model = XGBClassifier(tree_method='hist', random_state=42,
                                eval_metric='logloss', enable_categorical=False)

elif ROUND_CONFIG['model_type'] == 'random_forest':
    param_dist = {
        'n_estimators': [100, 200, 300, 400, 500],
        'max_depth': [3, 4, 5, 6, 8, 10, 12, None],
        'min_samples_split': [2, 3, 5, 8, 10, 15],
        'min_samples_leaf': [1, 2, 3, 4, 5, 8],
        'max_features': [0.3, 0.4, 0.5, 0.6, 0.7, 'sqrt', 'log2'],
        'max_samples': [0.5, 0.6, 0.7, 0.8, 0.9, None],
        'bootstrap': [True],
    }
    base_model = RandomForestClassifier(random_state=42, n_jobs=-1)

else:  # logistic_regression
    from sklearn.linear_model import LogisticRegression
    param_dist = {
        'C': [0.001, 0.01, 0.1, 0.5, 1.0, 5.0, 10.0, 50.0],
        'penalty': ['l1', 'l2'],
        'solver': ['liblinear', 'saga'],
        'max_iter': [1000],
    }
    base_model = LogisticRegression(random_state=42)

print(f"\nRunning RandomizedSearchCV (200 iterations, 5-fold CV)...")
search = RandomizedSearchCV(
    base_model, param_dist, n_iter=200, cv=cv, scoring='roc_auc',
    random_state=42, n_jobs=-1, verbose=0
)
search.fit(X_train_s[chosen_features], y_train)

best = search.best_estimator_
tr_p = best.predict_proba(X_train_s[chosen_features])[:, 1]
te_p = best.predict_proba(X_test_s[chosen_features])[:, 1]

print(f"\nBest params:")
for k, v in search.best_params_.items():
    print(f"  {k}: {v}")

print(f"\nResults:")
print(f"  CV AUC:       {search.best_score_:.4f}")
print(f"  Train AUC:    {roc_auc_score(y_train, tr_p):.4f}")
print(f"  Test AUC:     {roc_auc_score(y_test, te_p):.4f}")
print(f"  Train ACC:    {accuracy_score(y_train, best.predict(X_train_s[chosen_features])):.4f}")
print(f"  Test ACC:     {accuracy_score(y_test, best.predict(X_test_s[chosen_features])):.4f}")
print(f"  Brier (test): {brier_score_loss(y_test, te_p):.4f}")
print(f"  AUC gap:      {roc_auc_score(y_train, tr_p) - roc_auc_score(y_test, te_p):+.4f}")
print(f"  Prob range:   [{te_p.min():.3f}, {te_p.max():.3f}]")

if X_val is not None:
    val_p = best.predict_proba(X_val_s[chosen_features])[:, 1]
    print(f"  Val AUC:      {roc_auc_score(y_val, val_p):.4f}")
    print(f"  Val ACC:      {accuracy_score(y_val, best.predict(X_val_s[chosen_features])):.4f}")

# ============================================================================
# PART 4: REGULARIZATION TUNING (reduce overfitting)
# ============================================================================
print("\n" + "=" * 80)
print("PART 4: REGULARIZATION SWEEP (on best params)")
print("=" * 80)

if ROUND_CONFIG['model_type'] == 'random_forest':
    configs = [
        ('best params',           search.best_params_),
        ('depth=6, msl=3',       {**search.best_params_, 'max_depth': 6, 'min_samples_leaf': 3}),
        ('depth=5, msl=4',       {**search.best_params_, 'max_depth': 5, 'min_samples_leaf': 4}),
        ('depth=4, msl=5',       {**search.best_params_, 'max_depth': 4, 'min_samples_leaf': 5}),
        ('depth=3, msl=5, mf=0.7', {**search.best_params_, 'max_depth': 3, 'min_samples_leaf': 5, 'max_features': 0.7}),
    ]

    print(f"\n{'Config':>25s}  {'Train AUC':>10s}  {'Test AUC':>9s}  {'Gap':>7s}  {'Test ACC':>9s}  {'Prob Range':>15s}")
    print("-" * 85)

    for label, params in configs:
        rf = RandomForestClassifier(random_state=42, n_jobs=-1, **params)
        rf.fit(X_train_s[chosen_features], y_train)
        tr_p = rf.predict_proba(X_train_s[chosen_features])[:, 1]
        te_p = rf.predict_proba(X_test_s[chosen_features])[:, 1]
        tr_auc = roc_auc_score(y_train, tr_p)
        te_auc = roc_auc_score(y_test, te_p)
        te_acc = accuracy_score(y_test, rf.predict(X_test_s[chosen_features]))
        print(f"{label:>25s}    {tr_auc:.4f}     {te_auc:.4f}   {tr_auc-te_auc:+.4f}    {te_acc:.4f}   [{te_p.min():.3f}, {te_p.max():.3f}]")

elif ROUND_CONFIG['model_type'] == 'xgboost':
    configs = [
        ('best params',              search.best_params_),
        ('depth-1',                  {**search.best_params_, 'max_depth': max(2, search.best_params_.get('max_depth', 4) - 1)}),
        ('depth-2, more reg',        {**search.best_params_, 'max_depth': max(2, search.best_params_.get('max_depth', 4) - 2), 'reg_lambda': 7.0}),
        ('subsample=0.4, col=0.4',   {**search.best_params_, 'subsample': 0.4, 'colsample_bytree': 0.4}),
    ]

    print(f"\n{'Config':>30s}  {'Train AUC':>10s}  {'Test AUC':>9s}  {'Gap':>7s}  {'Test ACC':>9s}  {'Prob Range':>15s}")
    print("-" * 90)

    for label, params in configs:
        xgb = XGBClassifier(tree_method='hist', random_state=42,
                            eval_metric='logloss', enable_categorical=False, **params)
        xgb.fit(X_train_s[chosen_features], y_train)
        tr_p = xgb.predict_proba(X_train_s[chosen_features])[:, 1]
        te_p = xgb.predict_proba(X_test_s[chosen_features])[:, 1]
        tr_auc = roc_auc_score(y_train, tr_p)
        te_auc = roc_auc_score(y_test, te_p)
        te_acc = accuracy_score(y_test, xgb.predict(X_test_s[chosen_features]))
        print(f"{label:>30s}    {tr_auc:.4f}     {te_auc:.4f}   {tr_auc-te_auc:+.4f}    {te_acc:.4f}   [{te_p.min():.3f}, {te_p.max():.3f}]")

print("\n" + "=" * 80)
print("DONE — Copy your chosen features and params into men_generate_matchups.py")
print("=" * 80)

Round 1 FEATURE SELECTION & HYPERPARAMETER OPTIMIZATION
  350 games, 108 features
  Model type: xgboost

  Train: 243, Val: 54, Test: 53

PART 1: FEATURE SELECTION

--- Method 1: RF Gini Importance ---
  5man_bpm: 0.0828
  kenpom_rtg: 0.0476
  torvik_rtg: 0.0433
  3man_bpm: 0.0403
  wab: 0.0288
  5man_obpm: 0.0286
  3man_obpm: 0.0214
  5man_dbpm: 0.0211
  5man_dprpg: 0.0192
  experience_weighted_production: 0.0188
  5man_prpg: 0.0183
  3man_prpg: 0.0153
  size: 0.0149
  3man_dprpg: 0.0135
  def_lineup_depth_quality: 0.0128

--- Method 2: Permutation Importance ---
  5man_bpm: 0.0541
  kenpom_rtg: 0.0176
  3man_bpm: 0.0000
  wab: 0.0000
  torvik_rtg: 0.0000
  5man_prpg: 0.0000
  3man_prpg: 0.0000
  5man_dprpg: 0.0000
  3man_dprpg: 0.0000
  size: 0.0000
  height: 0.0000
  experience: 0.0000
  bench: 0.0000
  raw_tempo: 0.0000
  adj_tempo: 0.0000

--- Method 3: Mutual Information ---
  5man_bpm: 0.1876
  kenpom_rtg: 0.1773
  3man_bpm: 0.1575
  5man_prpg: 0.1505
  5man_dprpg: 0.1409
  5man